# Notebook principal — INF6243

Ce notebook exécute le pipeline complet et affiche les résultats clés.

In [1]:
from pathlib import Path
from IPython.display import display, Markdown, Image

import sys
sys.path.insert(0, "Code")
from notebook_workflow import (
    build_models_table,
    build_runs_comparison_table,
    load_report,
    run_all_configs,
)
from result_interpreter import interpret_report

In [2]:
# Strategie multi-runs: configurations definies dans le notebook.
def get_default_runs() -> dict:
    """Retourne les configurations de runs par défaut.

    Paramètres de configuration (appliqués dans chaque run):
        - max_samples (int | None): taille max de l'échantillon stratifié.
            Valeurs recommandées: None (100% du dataset) ou entier > 1000.
        - distilbert_epochs (int): nombre d'epochs pour DistilBERT.
            Range recommandé: 1 à 5 (1 rapide, 2-3 standard, >3 plus coûteux).
        - include_distilbert (bool): inclut ou non DistilBERT dans la comparaison.
            Valeurs: True ou False.
        - algorithm_switches (dict[str, bool]): activation fine de chaque algorithme.
            Ex: {"AdaBoost": False, "DistilBERT": True}. Si absent, tous les modèles sont actifs
            (avec `include_distilbert` comme compatibilité pour DistilBERT).
        - test_size (float): proportion du jeu de test.
            Range valide: 0.1 à 0.4 (strictement entre 0 et 1).
        - val_size (float): proportion du jeu de validation.
            Range valide: 0.05 à 0.3 (strictement entre 0 et 1).
        - cv_folds (int): nombre de folds pour la validation croisée.
            Range recommandé: 3 à 10.
        - scoring (str): métrique sklearn utilisée pour GridSearchCV.
            Valeurs typiques: "f1_macro", "accuracy", "precision_macro", "recall_macro".
        - model_param_overrides (dict[str, dict]): paramètres fixes par modèle.
            Exemple utile: `{"DistilBERT": {"epochs": 2, "batch_size": 8, "max_length": 160}}`.
            Impact: permet d'ajuster précisément le coût mémoire/temps et la capacité d'adaptation.
        - model_grid_overrides (dict[str, dict[str, list]]): surcharge des grilles GridSearch par modèle.
            Exemple utile: `{"MLPClassifier": {"clf__alpha": [1e-4, 1e-3]}}`.
            Impact: plus de combinaisons = meilleure chance d'optimum, mais coût CPU/RAM plus élevé.
        - selection_weights (tuple[float, float, float, float]): poids
            `(validation, test, cv, hate_recall)` du score de sélection final.
            Range recommandé: chaque poids entre 0 et 1, somme = 1.0.
        - hate_recall_floor (float): seuil minimum de recall pour `hate_speech` sur test.
            Range recommandé: 0.30 à 0.50.
        - hate_recall_penalty (float): pénalité soustraite au score si le seuil n'est pas atteint.
            Range recommandé: 0.01 à 0.05.
        - random_state (int): seed de reproductibilité.
            Valeur recommandée: entier positif (ex: 42).
        - DISTILBERT_PROXY_PENALTY (float): malus de prudence appliqué aux runs
            incluant DistilBERT quand `cv_fallback_for_models` contient DistilBERT.
            Range recommandé: 0.00 à 0.05.

    Notes:
        Le score de sélection est calculé par:
        `selection_score = w_val * val_f1_macro + w_test * test_f1_macro + w_cv * cv_f1_macro_mean + w_hate * hate_recall_test`.
        Si `hate_recall_test < hate_recall_floor`, une pénalité est soustraite.
        Plus `w_cv` est élevé, plus la stabilité inter-fold est valorisée.

    Retour:
        Dictionnaire des runs prêts à exécuter par `run_all_configs()`.
    """
    # Parametres communs (non mouvants entre les runs)
    TEST_SIZE = 0.2
    VAL_SIZE = 0.1
    SCORING = "f1_macro"
    RANDOM_STATE = 42

    # Overriding centralisé des hyperparamètres.
    # Utiliser `_BASE` si vous voulez garder les réglages model_zoo par défaut.
    MODEL_PARAM_OVERRIDES_BASE = {}
    MODEL_GRID_OVERRIDES_BASE = {}

    # DistilBERT: profils conseillés pour contrôler coût vs qualité.
    # - epochs: 1-3 pour itérations, 4-5 pour consolidation finale.
    # - batch_size: 8 si VRAM limitée, 16 compromis, 32 si GPU confortable.
    # - max_length: 128 défaut, 160/192 pour plus de contexte (coût en hausse).
    DISTILBERT_PARAMS_EP2 = {
        "DistilBERT": {
            "epochs": 2,
            "batch_size": 16,
            "max_length": 160,
            "learning_rate": 3e-5,
            "weight_decay": 0.01,
        }
    }
    DISTILBERT_PARAMS_EP3 = {
        "DistilBERT": {
            "epochs": 3,
            "batch_size": 16,
            "max_length": 160,
            "learning_rate": 3e-5,
            "weight_decay": 0.01,
        }
    }

    # MLP: exemple de grille étendue (plus coûteuse) pour exploration dédiée.
    MLP_GRID_WIDE = {
        "MLPClassifier": {
            "svd__n_components": [20, 40, 60],
            "clf__hidden_layer_sizes": [(128,), (256, 128), (256, 256)],
            "clf__alpha": [1e-4, 5e-4, 1e-3],
            "clf__learning_rate_init": [1e-3, 5e-4, 3e-4],
        }
    }

    # Parametres partages par sous-groupes de runs
    CV_FOLDS_STANDARD = 5
    CV_FOLDS_FAST = 3

    # Weights recommandés pour selection_score = val + test + cv + hate_recall
    # validation: garde-fou tuning, test: performance réelle, cv: stabilité, hate_recall: protection classe minoritaire.
    WEIGHTS_BALANCED = (0.30, 0.35, 0.20, 0.15)
    WEIGHTS_TEST_PRIORITY = (0.25, 0.45, 0.15, 0.15)

    # Weights additionnels pour scenarios méthodologiques
    WEIGHTS_METHOD_STRICT = (0.45, 0.20, 0.20, 0.15)
    WEIGHTS_CV_HEAVY = (0.25, 0.25, 0.35, 0.15)
    WEIGHTS_DISTILBERT_SAFE = (0.30, 0.40, 0.15, 0.15)

    # Garde-fou classe minoritaire hate_speech
    HATE_RECALL_FLOOR = 0.40
    HATE_RECALL_PENALTY = 0.03

    # Activation fine des algorithmes (paramètre run-level)
    ALGORITHM_SWITCHES_ALL = {
        "NaiveBayes": True,
        "LogisticRegression": True,
        "LinearSVC": True,
        "KNN": True,
        "DecisionTree": True,
        "RandomForest": True,
        "AdaBoost": True,
        "MLPClassifier": True,
        "DistilBERT": True,
    }
    ALGORITHM_SWITCHES_CLASSIC_ONLY = {
        **ALGORITHM_SWITCHES_ALL,
        "DistilBERT": False,
    }

    return {
        # Baselines
        "run_a_data_balance": {
            "why": (
                "Run A = baseline robuste sur plus de donnees. "
                "On utilise 100% des echantillons pour mieux couvrir la classe minoritaire hate_speech."
            ),
            "config": {
                "max_samples": None,
                "distilbert_epochs": 1,
                "include_distilbert": True,
                "test_size": TEST_SIZE,
                "val_size": VAL_SIZE,
                "cv_folds": CV_FOLDS_STANDARD,
                "scoring": SCORING,
                "model_param_overrides": MODEL_PARAM_OVERRIDES_BASE,
                "model_grid_overrides": MODEL_GRID_OVERRIDES_BASE,
                "selection_weights": WEIGHTS_BALANCED,
                "algorithm_switches": ALGORITHM_SWITCHES_ALL,
                "hate_recall_floor": HATE_RECALL_FLOOR,
                "hate_recall_penalty": HATE_RECALL_PENALTY,
                "random_state": RANDOM_STATE,
            },
        },
       # Ligue classique
        "run_e_method_strict_classic": {
            "why": "Run E = sélection méthodologique stricte, test faiblement pondéré, classiques uniquement.",
            "config": {
                "max_samples": None,
                "distilbert_epochs": 1,
                "include_distilbert": False,
                "test_size": TEST_SIZE,
                "val_size": VAL_SIZE,
                "cv_folds": CV_FOLDS_STANDARD,
                "scoring": SCORING,
                "model_param_overrides": MODEL_PARAM_OVERRIDES_BASE,
                "model_grid_overrides": MODEL_GRID_OVERRIDES_BASE,
                "selection_weights": WEIGHTS_METHOD_STRICT,
                "algorithm_switches": ALGORITHM_SWITCHES_CLASSIC_ONLY,
                "hate_recall_floor": HATE_RECALL_FLOOR,
                "hate_recall_penalty": HATE_RECALL_PENALTY,
                "random_state": RANDOM_STATE,
            },
        },
        # "run_f_cv_heavy_classic": {
        #     "why": "Run F = robustesse prioritaire via CV fort, classiques uniquement.",
        #     "config": {
        #         "max_samples": None,
        #         "distilbert_epochs": 1,
        #         "include_distilbert": False,
        #         "test_size": TEST_SIZE,
        #         "val_size": VAL_SIZE,
        #         "cv_folds": CV_FOLDS_STANDARD,
        #         "scoring": SCORING,
        #         "model_param_overrides": MODEL_PARAM_OVERRIDES_BASE,
        #         # Run F: on élargit la grille MLP pour explorer davantage en mode classique.
        #         "model_grid_overrides": MLP_GRID_WIDE,
        #         "selection_weights": WEIGHTS_CV_HEAVY,
        #         "algorithm_switches": ALGORITHM_SWITCHES_CLASSIC_ONLY,
        #         "hate_recall_floor": HATE_RECALL_FLOOR,
        #         "hate_recall_penalty": HATE_RECALL_PENALTY,
        #         "random_state": RANDOM_STATE,
        #     },
        # },
        

        # Ligue étendue avec DistilBERT (weights adaptés au fallback CV proxy)
        "run_g_distilbert_safe_ep3": {
            "why": "Run G = DistilBERT (3 epochs) avec même compensation de pondération CV.",
            "config": {
                "max_samples": None,
                "distilbert_epochs": 3,
                "include_distilbert": True,
                "test_size": TEST_SIZE,
                "val_size": VAL_SIZE,
                "cv_folds": CV_FOLDS_STANDARD,
                "scoring": SCORING,
                # Run G: profil DistilBERT explicite pour pilotage reproductible.
                "model_param_overrides": DISTILBERT_PARAMS_EP3,
                "model_grid_overrides": MODEL_GRID_OVERRIDES_BASE,
                "selection_weights": WEIGHTS_DISTILBERT_SAFE,
                "algorithm_switches": ALGORITHM_SWITCHES_ALL,
                "hate_recall_floor": HATE_RECALL_FLOOR,
                "hate_recall_penalty": HATE_RECALL_PENALTY,
                "random_state": RANDOM_STATE,
            },
        },
    }


def get_exhaustive_runs() -> dict:
    """Construit une matrice exhaustive de runs pour DistilBERT/MLP/AdaBoost.

    Stratégie:
        - couvrir des profils de complexité croissante pour les 3 modèles;
        - générer toutes les combinaisons (produit cartésien) pour faciliter
          une exploration systématique avant affinement fin;
        - conserver une politique de scoring unique pour comparer équitablement.

    Attention coût:
        Cette matrice est volontairement non conservative sur CPU/RAM/VRAM.
        Certains profils DistilBERT (batch_size élevé + max_length élevé)
        peuvent dépasser la mémoire disponible selon le GPU.
    """
    base_runs = get_default_runs()

    # Constantes de run pour éviter les valeurs en dur répétées.
    EXH_MAX_SAMPLES = None
    EXH_TEST_SIZE = 0.2
    EXH_VAL_SIZE = 0.1
    EXH_CV_FOLDS = 5
    EXH_SCORING = "f1_macro"
    EXH_SELECTION_WEIGHTS = (0.30, 0.40, 0.15, 0.15)
    EXH_HATE_RECALL_FLOOR = 0.40
    EXH_HATE_RECALL_PENALTY = 0.03
    EXH_RANDOM_STATE = 42
    EXH_ALGORITHM_SWITCHES = {
        "NaiveBayes": True,
        "LogisticRegression": True,
        "LinearSVC": True,
        "KNN": True,
        "DecisionTree": True,
        "RandomForest": True,
        "AdaBoost": True,
        "MLPClassifier": True,
        "DistilBERT": True,
    }

    # Profils DistilBERT (du plus rapide au plus gourmand VRAM).
    distilbert_profiles: dict[str, dict[str, dict[str, float | int]]] = {
        "balanced": {
            "DistilBERT": {
                "epochs": 2,
                "batch_size": 16,
                "max_length": 160,
                "learning_rate": 3e-5,
                "weight_decay": 0.01,
            }
        },
        "fast": {
            "DistilBERT": {
                "epochs": 1,
                "batch_size": 16,
                "max_length": 128,
                "learning_rate": 5e-5,
                "weight_decay": 0.01,
            }
        },
        "robust": {
            "DistilBERT": {
                "epochs": 3,
                "batch_size": 32,
                "max_length": 224,
                "learning_rate": 2e-5,
                "weight_decay": 0.02,
            }
        },
        "vram_max": {
            "DistilBERT": {
                "epochs": 4,
                "batch_size": 48,
                "max_length": 320,
                "learning_rate": 2e-5,
                "weight_decay": 0.02,
            }
        },
    }

    # Profils MLP (grilles de plus en plus larges / coûteuses).
    mlp_profiles: dict[str, dict[str, dict[str, list]]] = {
        "balanced": {
            "MLPClassifier": {
                "svd__n_components": [20, 40, 60],
                "clf__hidden_layer_sizes": [(128,), (256, 128)],
                "clf__alpha": [1e-4, 5e-4, 1e-3],
                "clf__learning_rate_init": [1e-3, 5e-4],
            }
        },
        "fast": {
            "MLPClassifier": {
                "svd__n_components": [20, 40],
                "clf__hidden_layer_sizes": [(128,)],
                "clf__alpha": [1e-4, 1e-3],
                "clf__learning_rate_init": [1e-3],
            }
        },
        "ultra": {
            "MLPClassifier": {
                "svd__n_components": [40, 80, 120],
                "clf__hidden_layer_sizes": [(256, 128), (256, 256), (512, 256)],
                "clf__alpha": [1e-6, 1e-5, 1e-4, 5e-4],
                "clf__learning_rate_init": [1e-3, 8e-4, 5e-4, 2e-4],
            }
        },
        "wide": {
            "MLPClassifier": {
                "svd__n_components": [20, 40, 60, 80],
                "clf__hidden_layer_sizes": [(128,), (256, 128), (256, 256)],
                "clf__alpha": [1e-5, 1e-4, 5e-4, 1e-3],
                "clf__learning_rate_init": [1e-3, 7e-4, 5e-4, 3e-4],
            }
        },
    }

    # Profils AdaBoost (de standard à très coûteux).
    adaboost_profiles: dict[str, dict[str, dict[str, list]]] = {
        "balanced": {
            "AdaBoost": {
                "clf__n_estimators": [200, 400, 800],
                "clf__learning_rate": [0.03, 0.1, 0.3],
                "clf__estimator__max_depth": [1, 2, 3],
                "clf__estimator__min_samples_leaf": [1, 2],
            }
        },
        "fast": {
            "AdaBoost": {
                "clf__n_estimators": [200, 400],
                "clf__learning_rate": [0.05, 0.1],
                "clf__estimator__max_depth": [1, 2],
                "clf__estimator__min_samples_leaf": [1, 2],
            }
        },
        "ultra": {
            "AdaBoost": {
                "clf__n_estimators": [800, 1200, 1600],
                "clf__learning_rate": [0.01, 0.03, 0.1, 0.3, 0.6],
                "clf__estimator__max_depth": [2, 3, 4, 5],
                "clf__estimator__min_samples_leaf": [1, 2, 4, 8],
            }
        },
        "wide": {
            "AdaBoost": {
                "clf__n_estimators": [400, 800, 1200],
                "clf__learning_rate": [0.01, 0.03, 0.1, 0.3],
                "clf__estimator__max_depth": [1, 2, 3, 4],
                "clf__estimator__min_samples_leaf": [1, 2, 4],
            }
        },
    }

    exhaustive_runs: dict[str, dict] = {}
    # On garde un baseline existant pour référence de comparaison.
    if "run_a_data_balance" in base_runs:
        exhaustive_runs["run_a_data_balance"] = base_runs["run_a_data_balance"]
    
    def _single_model_switches(target_model: str) -> dict[str, bool]:
        """Active uniquement le modèle ciblé dans un run."""
        return {name: (name == target_model) for name in EXH_ALGORITHM_SWITCHES}

    # Mapping explicite: 1 run = 1 profil DistilBERT.
    distilbert_run_map = {
        "run_d_fast": "fast",
        "run_d_balanced": "balanced",
        "run_d_robust": "robust",
        "run_d_vram_max": "vram_max",
    }
    for run_name, profile_name in distilbert_run_map.items():
        d_params = distilbert_profiles[profile_name]
        exhaustive_runs[run_name] = {
            "why": f"Run tuning DistilBERT seul (profil={profile_name}).",
            "config": {
                "max_samples": EXH_MAX_SAMPLES,
                "distilbert_epochs": int(d_params["DistilBERT"]["epochs"]),
                "include_distilbert": True,
                "test_size": EXH_TEST_SIZE,
                "val_size": EXH_VAL_SIZE,
                "cv_folds": EXH_CV_FOLDS,
                "scoring": EXH_SCORING,
                "model_param_overrides": d_params,
                "model_grid_overrides": {},
                "selection_weights": EXH_SELECTION_WEIGHTS,
                "algorithm_switches": _single_model_switches("DistilBERT"),
                "hate_recall_floor": EXH_HATE_RECALL_FLOOR,
                "hate_recall_penalty": EXH_HATE_RECALL_PENALTY,
                "random_state": EXH_RANDOM_STATE,
            },
        }

    # Mapping explicite: 1 run = 1 profil MLP.
    mlp_run_map = {
        "run_mlp_fast": "fast",
        "run_mlp_balanced": "balanced",
        "run_mlp_wide": "wide",
        "run_mlp_ultra": "ultra",
    }
    for run_name, profile_name in mlp_run_map.items():
        m_grid = mlp_profiles[profile_name]
        exhaustive_runs[run_name] = {
            "why": f"Run tuning MLP seul (profil={profile_name}).",
            "config": {
                "max_samples": EXH_MAX_SAMPLES,
                "distilbert_epochs": 1,
                "include_distilbert": False,
                "test_size": EXH_TEST_SIZE,
                "val_size": EXH_VAL_SIZE,
                "cv_folds": EXH_CV_FOLDS,
                "scoring": EXH_SCORING,
                "model_param_overrides": {},
                "model_grid_overrides": m_grid,
                "selection_weights": EXH_SELECTION_WEIGHTS,
                "algorithm_switches": _single_model_switches("MLPClassifier"),
                "hate_recall_floor": EXH_HATE_RECALL_FLOOR,
                "hate_recall_penalty": EXH_HATE_RECALL_PENALTY,
                "random_state": EXH_RANDOM_STATE,
            },
        }

    # Mapping explicite: 1 run = 1 profil AdaBoost.
    adaboost_run_map = {
        "run_ada_fast": "fast",
        "run_ada_balanced": "balanced",
        "run_ada_wide": "wide",
        "run_ada_ultra": "ultra",
    }
    for run_name, profile_name in adaboost_run_map.items():
        a_grid = adaboost_profiles[profile_name]
        exhaustive_runs[run_name] = {
            "why": f"Run tuning AdaBoost seul (profil={profile_name}).",
            "config": {
                "max_samples": EXH_MAX_SAMPLES,
                "distilbert_epochs": 1,
                "include_distilbert": False,
                "test_size": EXH_TEST_SIZE,
                "val_size": EXH_VAL_SIZE,
                "cv_folds": EXH_CV_FOLDS,
                "scoring": EXH_SCORING,
                "model_param_overrides": {},
                "model_grid_overrides": a_grid,
                "selection_weights": EXH_SELECTION_WEIGHTS,
                "algorithm_switches": _single_model_switches("AdaBoost"),
                "hate_recall_floor": EXH_HATE_RECALL_FLOOR,
                "hate_recall_penalty": EXH_HATE_RECALL_PENALTY,
                "random_state": EXH_RANDOM_STATE,
            },
        }

    return exhaustive_runs


# Active la version exhaustive pour parcourir l'ensemble des profils.
RUNS = get_exhaustive_runs()
# RUNS = get_default_runs()

# Regle paramétrable de compensation DistilBERT (CV proxy).
# Valeurs utiles: 0.00 (sans malus) a 0.05 (malus prudent fort).
DISTILBERT_PROXY_PENALTY = 0.02

workflow = run_all_configs(RUNS, distilbert_proxy_penalty=DISTILBERT_PROXY_PENALTY)
run_summary_df = workflow["run_summary_df"]
BEST_RUN = workflow["best_run"]
artifacts = workflow["artifacts"]
FIGURE_NAMES = workflow["figure_names"]

# Affichage clair: score brut vs score ajusté + indicateurs DistilBERT proxy
display(
    run_summary_df[
        [
            "run",
            "best_model",
            "include_distilbert",
            "distilbert_cv_proxy",
            "fairness_penalty",
            "best_selection_score",
            "adjusted_selection_score",
            "best_test_f1_macro",
        ]
    ]
)
print(f"Run de reference selectionne pour la suite (score ajusté): {BEST_RUN}")
artifacts

Execution: run_a_data_balance
Pourquoi: Run A = baseline robuste sur plus de donnees. On utilise 100% des echantillons pour mieux couvrir la classe minoritaire hate_speech.
Configuration: {'max_samples': None, 'distilbert_epochs': 1, 'include_distilbert': True, 'test_size': 0.2, 'val_size': 0.1, 'cv_folds': 5, 'scoring': 'f1_macro', 'model_param_overrides': {}, 'model_grid_overrides': {}, 'selection_weights': (0.3, 0.35, 0.2, 0.15), 'algorithm_switches': {'NaiveBayes': True, 'LogisticRegression': True, 'LinearSVC': True, 'KNN': True, 'DecisionTree': True, 'RandomForest': True, 'AdaBoost': True, 'MLPClassifier': True, 'DistilBERT': True}, 'hate_recall_floor': 0.4, 'hate_recall_penalty': 0.03, 'random_state': 42}
Device détecté (pour deep learning): cuda
/usr/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the c

RuntimeError: Run 'run_d_fast' en échec via subprocess: Aucun modèle n'a pu être entraîné. Vérifier les dépendances et les données.

## Pourquoi une matrice explicite de runs ?

- **Objectif**: chercher la meilleure configuration de chaque modèle complexe séparément.
- **Couverture**: 1 run = 1 profil (`DistilBERT` seul, `MLPClassifier` seul, `AdaBoost` seul), plus un baseline.
- **Approche non conservative**: profils mémoire/VRAM agressifs (batch size, max_length, grilles larges) pour pousser la recherche.
- **Affinage ensuite**: après ce balayage, conserver les meilleurs profils par modèle et raffiner localement.
- **Comparaison équitable**: même politique de scoring, avec compensation DistilBERT CV proxy via `DISTILBERT_PROXY_PENALTY`.

Cette stratégie évite le produit cartésien coûteux et garde une lecture claire des effets de chaque famille de paramètres.

In [ ]:
report = load_report(Path(artifacts["report_path"]))

print("Meilleur modèle:", report.get("best_model"))
print("Métriques test:", report.get("best_model_test_metrics"))
print("DistilBERT:", report.get("distilbert_note"))
print("Configuration:", report.get("run_config"))
display(Markdown("**Justification:** " + report.get("best_model_selection_explanation", "")))

In [ ]:
# Tableau complet: tous les modèles, statut, métriques, hyperparamètres
build_models_table(report)

In [ ]:
# Interpreteur de resultats (script dedie)
interpreter_summary = interpret_report(report, weak_f1_threshold=0.55)
interpreter_summary

## Pourquoi `best_cv_score` peut être vide pour DistilBERT ?

- Les modèles classiques utilisent `GridSearchCV`, donc un `best_cv_score` est produit.
- DistilBERT est entraîné en fine-tuning direct (pas de GridSearchCV complet pour limiter le coût).
- Le fallback de robustesse est documenté dans `model_selection_method.cv_fallback_for_models`.

In [ ]:
# Comparaison inter-runs (avec compensation DistilBERT CV proxy)
reports_dir = Path(artifacts["report_path"]).parent
comparison_df = build_runs_comparison_table(
    reports_dir,
    distilbert_proxy_penalty=DISTILBERT_PROXY_PENALTY,
)

if comparison_df.empty:
    print("Aucun report multi-run detecte.")
else:
    ordered_cols = [
        "run",
        "best_model",
        "include_distilbert",
        "distilbert_cv_proxy",
        "fairness_penalty",
        "best_selection_score",
        "adjusted_selection_score",
        "best_test_f1_macro",
        "distilbert_note",
    ]
    display(comparison_df[ordered_cols])

    print("\n=== Ligue classique (sans DistilBERT) ===")
    classic_df = comparison_df[comparison_df["include_distilbert"] == False]
    if classic_df.empty:
        print("Aucun run classique trouvé.")
    else:
        display(classic_df[ordered_cols])

    print("\n=== Ligue étendue (avec DistilBERT) ===")
    distil_df = comparison_df[comparison_df["include_distilbert"] == True]
    if distil_df.empty:
        print("Aucun run DistilBERT trouvé.")
    else:
        display(distil_df[ordered_cols])

    print("Astuce: la comparaison est aussi disponible en figure (runs_comparison_overview.png).")

In [ ]:
fig_dir = Path(artifacts["figures_dir"])
for fig_name in FIGURE_NAMES:
    fig_path = fig_dir / fig_name
    if fig_path.exists():
        display(Markdown(f"### {fig_name}"))
        display(Image(filename=str(fig_path)))
    else:
        print(f"Figure absente: {fig_path}")